# Modern LLM Techniques: from our tiny transformer to frontier-scale models

Companion notebook to the bonus lesson of **Transformers From Scratch**. In the main notebooks we
built a ~450K-parameter transformer. The largest open-weight models are over a million times larger
(DeepSeek-V3: 671B parameters), and they rely on a set of techniques our model didn't need. Each
section below is a small, runnable implementation of one of them:

1. Scaling laws: how big, and how much data
2. RMSNorm
3. SwiGLU feed-forward
4. Grouped-query attention and the KV cache
5. The FlashAttention idea: online softmax
6. Mixture of Experts (MoE) with load balancing
7. Speculative decoding
8. Quantization
9. Post-training: preference optimization (DPO)
10. Superposition and sparse autoencoders (interpretability)

**Sources and scope.** Labs such as Anthropic, OpenAI and Google do not publish full architecture
details of their frontier models. Every factual claim here is taken from a public paper, model card
or released configuration file, cited inline and listed under *References*. Where Anthropic is
mentioned, it is for published research (Constitutional AI and interpretability), not for the
architecture of Claude.

**Reproducibility.** All experiments use fixed seeds and run on a laptop CPU in about half a minute.
Experiments that involve training are repeated over 5 seeds and reported as mean ± standard
deviation. Wall-clock timings vary between machines and runs. The experiments are deliberately tiny:
they illustrate mechanisms and are not benchmarks.

In [1]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_num_threads(4)
print("torch", torch.__version__)

torch 2.6.0


## 1. Scaling laws

Kaplan et al. (2020) found that the cross-entropy loss of language models falls as a power law in
model size, dataset size and the amount of training compute. Hoffmann et al. (2022, "Chinchilla") revisited how a fixed compute
budget should be split between parameters $N$ and training tokens $D$, using the approximation
$C \approx 6ND$ FLOPs. Their three estimation approaches agree that $N$ and $D$ should grow in about
equal proportion: "for every doubling of model size the number of training tokens should also be
doubled". Their Table 3 places the optimum near **20 tokens per parameter** over the range they
tabulate (for example 67B parameters and 1.5T tokens at $5.76 \times 10^{23}$ FLOPs), and Chinchilla
itself was trained with 70B parameters on 1.4T tokens. By this standard they conclude that earlier
large models were "significantly undertrained".

Their third approach fits a parametric loss

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}} .$$

Minimising it subject to $6ND = C$ has a closed form:

$$N_{opt} = G\,(C/6)^{a}, \qquad D_{opt} = G^{-1}\,(C/6)^{b}, \qquad
G = \left(\frac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}, \quad
a = \frac{\beta}{\alpha+\beta}, \quad b = \frac{\alpha}{\alpha+\beta} .$$

**Caveat.** Besiroglu et al. (2024) showed that the constants reported for this third approach are
inconsistent with the other two: they imply a much higher ratio ("approximately 70 tokens per
parameter"), not 20. Re-fitting the same functional form to the data, they obtain constants that
again imply "around 20 tokens per parameter". The cell below evaluates the closed form for both
sets of constants. The fitted loss values refer to Chinchilla's training data and are not
comparable across datasets.

In [2]:
def train_flops(N, D):
    return 6 * N * D           # ~2 FLOPs/param/token forward + ~4 backward (Kaplan et al., 2020)

# Public figures: (parameters, training tokens)
models = {
    "GPT-3 (2020)":          (175e9, 300e9),     # Brown et al.: 300 billion tokens
    "Chinchilla (2022)":     (70e9,  1.4e12),    # Hoffmann et al., Table 1
    "Llama 2 70B (2023)":    (70e9,  2.0e12),    # Touvron et al.: 2 trillion tokens
    "Llama 3 8B (2024)":     (8e9,   15e12),     # model card: "15T+" tokens, so a lower bound
    "Llama 3.1 405B (2024)": (405e9, 15.6e12),   # Grattafiori et al.
}
print(f"{'model':22s} {'tokens/param':>12s} {'train FLOPs':>12s}")
for name, (N, D) in models.items():
    print(f"{name:22s} {D / N:12,.1f} {train_flops(N, D):12.1e}")
print("check: Grattafiori et al. report 3.8e25 FLOPs for the 405B model, which equals 6ND above.")

# Parametric fit L(N, D) = E + A / N**alpha + B / D**beta  (Hoffmann et al., Approach 3).
# The first row uses the constants as reported (Eq. 10), with the extra digits that
# Besiroglu et al. recovered from the paper's TeX source; the second is their re-fit (Eq. 3).
FITS = {
    "Hoffmann et al., as reported": (1.6934, 406.4, 410.7, 0.3392, 0.2849),
    "Besiroglu et al., re-fit":     (1.8172, 482.01, 2085.43, 0.3478, 0.3658),
}

def optimal_split(C, E, A, B, alpha, beta):
    # minimise L(N, D) subject to 6 N D = C
    G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
    N = G * (C / 6) ** (beta / (alpha + beta))
    D = C / (6 * N)
    return N, D, E + A / N**alpha + B / D**beta

for name, fit in FITS.items():
    print(f"\n{name}: N_opt grows as C^{fit[4] / (fit[3] + fit[4]):.3f}")
    for C in [1e21, 1e22, 1e23, 5.76e23, 1e25]:
        N, D, L = optimal_split(C, *fit)
        print(f"  C = {C:.2e} FLOPs: N = {N / 1e9:6.1f} B params | D = {D / 1e9:7.0f} B tokens | "
              f"{D / N:5.1f} tokens/param | predicted loss {L:.3f}")

# sanity check of the closed form against a brute-force search over N
C, fit = 5.76e23, FITS["Besiroglu et al., re-fit"]
grid = torch.logspace(9, 13, 20001, dtype=torch.float64)
loss = fit[0] + fit[1] / grid**fit[3] + fit[2] / (C / (6 * grid))**fit[4]
print(f"\nbrute-force optimum at C = 5.76e23: N = {grid[loss.argmin()].item() / 1e9:.1f} B params "
      f"(closed form: {optimal_split(C, *fit)[0] / 1e9:.1f} B)")

model                  tokens/param  train FLOPs
GPT-3 (2020)                    1.7      3.2e+23
Chinchilla (2022)              20.0      5.9e+23
Llama 2 70B (2023)             28.6      8.4e+23
Llama 3 8B (2024)           1,875.0      7.2e+23
Llama 3.1 405B (2024)          38.5      3.8e+25
check: Grattafiori et al. report 3.8e25 FLOPs for the 405B model, which equals 6ND above.

Hoffmann et al., as reported: N_opt grows as C^0.456
  C = 1.00e+21 FLOPs: N =    2.2 B params | D =      75 B tokens |  34.0 tokens/param | predicted loss 2.295
  C = 1.00e+22 FLOPs: N =    6.3 B params | D =     263 B tokens |  41.5 tokens/param | predicted loss 2.115
  C = 1.00e+23 FLOPs: N =   18.1 B params | D =     920 B tokens |  50.7 tokens/param | predicted loss 1.988
  C = 5.76e+23 FLOPs: N =   40.3 B params | D =    2382 B tokens |  59.1 tokens/param | predicted loss 1.918
  C = 1.00e+25 FLOPs: N =  148.3 B params | D =   11235 B tokens |  75.7 tokens/param | predicted loss 1.838

Besiroglu et al.

**Reading the output.** $5.76 \times 10^{23}$ FLOPs is the budget of Gopher and Chinchilla. At that
budget the re-fitted law puts the optimum at 72B parameters and 1.3T tokens, close to Chinchilla's
actual 70B and 1.4T, and its ratio stays between 17 and 22 tokens per parameter over four orders of
magnitude of compute. The constants as reported give a smaller model (40B) and a ratio that grows
with the budget, from 34 to 76 over the same range.

GPT-3 saw 1.7 tokens per parameter, far below either estimate. Modern open models go the other way:
Llama 3 8B saw at least 1,875 tokens per parameter (its token count is reported only as "15T+").
This is deliberate. Grattafiori et al. (2024)
state that their 405B flagship is "an approximately compute-optimal size for our training budget",
while "we also train our smaller models for much longer than is compute-optimal. The resulting models
perform better than compute-optimal models at the same inference budget." Sardana et al. (2024) make
the same point formally: when inference demand is large, the cost-optimal model is smaller and
trained longer than the Chinchilla optimum.

## 2. RMSNorm

Our model used LayerNorm, which subtracts the mean and divides by the standard deviation. RMSNorm
(Zhang & Sennrich, 2019) drops the re-centring and only rescales by the root mean square:

$$\mathrm{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \odot g .$$

It has a gain $g$ but no bias, so half the parameters of LayerNorm. The authors report "comparable
performance against LayerNorm" while reducing running time "by 7%~64% on different models". The
released configurations of Llama 2 and 3, Mistral 7B, Mixtral and DeepSeek-V3 all use RMSNorm.

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms_inv = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms_inv * self.weight

x = torch.randn(8, 32, 512) * 3 + 2          # activations with non-zero mean and large scale
for name, norm in [("LayerNorm", nn.LayerNorm(512)), ("RMSNorm", RMSNorm(512))]:
    y = norm(x)
    print(f"{name:9s} params {sum(p.numel() for p in norm.parameters()):5d} | "
          f"out mean {y.mean():+.3f} | out RMS {y.pow(2).mean().sqrt():.3f}")

LayerNorm params  1024 | out mean -0.000 | out RMS 1.000
RMSNorm   params   512 | out mean +0.552 | out RMS 1.000


## 3. SwiGLU feed-forward

Our FFN was `Linear -> GELU -> Linear` with a 4x expansion. Shazeer (2020) tested gated variants
in which one projection passes through a nonlinearity and multiplies a second projection
element-wise, and found that "some of them yield quality improvements over the typically-used ReLU
or GELU activations". The SwiGLU variant is

$$\mathrm{FFN}_{\mathrm{SwiGLU}}(x) = \big(\mathrm{Swish}_1(xW) \otimes xV\big)\,W_2 ,$$

where $\mathrm{Swish}_1$ is SiLU. It has three weight matrices instead of two, so the hidden size is
reduced "by a factor of 2/3" to keep the parameter count constant. Llama 3 lists SwiGLU as its
activation function (Grattafiori et al., 2024, Table 3).

In [4]:
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w_gate = nn.Linear(dim, hidden, bias=False)
        self.w_up = nn.Linear(dim, hidden, bias=False)
        self.w_down = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

d = 512
gelu_ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
swiglu = SwiGLU(d, int(8 * d / 3))          # 2/3 * 4d hidden units
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"GELU FFN  (hidden {4 * d}): {count(gelu_ffn):,} params")
print(f"SwiGLU    (hidden {int(8 * d / 3)}): {count(swiglu):,} params")

GELU FFN  (hidden 2048): 2,099,712 params
SwiGLU    (hidden 1365): 2,096,640 params


## 4. Grouped-query attention and the KV cache

During generation, each new token attends to all previous tokens. Their keys and values do not
change, so inference engines compute them once and store them: the **KV cache**. For one sequence
of $T$ tokens it holds

$$\text{bytes} = 2 \cdot L \cdot h_{kv} \cdot d_{head} \cdot T \cdot b$$

for $L$ layers, $h_{kv}$ key/value heads of dimension $d_{head}$ and $b$ bytes per number. At long
context lengths this often becomes the dominant memory cost.

**Multi-query attention** (Shazeer, 2019) shares a single key/value head across all query heads.
**Grouped-query attention** (Ainslie et al., 2023) is the intermediate case: groups of query heads
share a key/value head, achieving "quality close to multi-head attention with comparable speed to
MQA". Llama 2 70B has 80 layers and 64 query heads with 8 key/value heads (released configuration);
Llama 3 uses 8 key/value heads at every size (Grattafiori et al., 2024).

In [5]:
class GQAttention(nn.Module):
    # n_kv_heads == n_heads -> multi-head attention, n_kv_heads == 1 -> multi-query attention
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.h, self.kv, self.hd = n_heads, n_kv_heads, dim // n_heads
        self.wq = nn.Linear(dim, n_heads * self.hd, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.hd, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.hd, bias=False)
        self.wo = nn.Linear(n_heads * self.hd, dim, bias=False)

    def forward(self, x, cache=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        if cache is not None:                      # reuse keys/values of earlier tokens
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        rep = self.h // self.kv                    # each K/V head serves `rep` query heads
        k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        assert T == 1 or k.shape[2] == T           # prefill, or decode one token at a time
        out = F.scaled_dot_product_attention(q, k, v, is_causal=(k.shape[2] == T))
        return self.wo(out.transpose(1, 2).reshape(B, T, -1))

# Positional encoding is left out: the equivalence checked below does not depend on it.
attn = GQAttention(dim=64, n_heads=8, n_kv_heads=2).eval()
x = torch.randn(1, 10, 64)
with torch.no_grad():
    full = attn(x)                                  # all 10 positions in one pass
    cache, steps = {}, []
    for t in range(10):                             # one token at a time, with the cache
        steps.append(attn(x[:, t:t + 1], cache))
print("max |full - cached|:", f"{(full - torch.cat(steps, 1)).abs().max().item():.1e}")
print("cached K shape:", tuple(cache["k"].shape), "(batch, kv_heads, tokens, head_dim)")

max |full - cached|: 2.1e-07
cached K shape: (1, 2, 10, 8) (batch, kv_heads, tokens, head_dim)


In [6]:
# How much faster is decoding with a cache? A 4-layer attention stack, generating 384 tokens.
# Wall-clock time on CPU, one run: illustrative, and different on every machine.
layers = nn.ModuleList(GQAttention(256, 8, 2) for _ in range(4)).eval()
seq = torch.randn(1, 384, 256)

def decode(use_cache):
    caches = [{} for _ in layers]
    times = []
    with torch.no_grad():
        for t in range(seq.shape[1]):
            t0 = time.perf_counter()
            if use_cache:
                h = seq[:, t:t + 1]
                for layer, c in zip(layers, caches):
                    h = h + layer(h, c)
            else:                                   # recompute the whole prefix every step
                h = seq[:, : t + 1]
                for layer in layers:
                    h = h + layer(h)
            times.append(time.perf_counter() - t0)
    return torch.tensor(times)

no_cache, with_cache = decode(False), decode(True)
print(f"without cache: {no_cache.sum():.2f} s total, last token {1e3 * no_cache[-20:].mean():.2f} ms")
print(f"with cache:    {with_cache.sum():.2f} s total, last token {1e3 * with_cache[-20:].mean():.2f} ms")
print("ms per token (every 32nd step; timings vary from run to run):")
print("  no cache:", [round(1e3 * v, 2) for v in no_cache[::32].tolist()])
print("  cache:   ", [round(1e3 * v, 2) for v in with_cache[::32].tolist()])

without cache: 0.51 s total, last token 3.01 ms
with cache:    0.07 s total, last token 0.24 ms
ms per token (every 32nd step; timings vary from run to run):
  no cache: [0.19, 0.27, 0.46, 0.59, 0.73, 0.95, 1.22, 1.44, 1.77, 2.01, 2.37, 2.78]
  cache:    [0.18, 0.14, 0.17, 0.17, 0.18, 0.19, 0.18, 0.2, 0.19, 0.21, 0.23, 0.23]


In [7]:
def kv_cache_gib(layers, kv_heads, head_dim, tokens, bytes_per=2):
    return 2 * layers * kv_heads * head_dim * tokens * bytes_per / 1024**3   # 2 = keys + values

# Llama-2-70B shape (released config): 80 layers, 64 query heads, head_dim 128, fp16 cache.
# Llama 2 itself has a 4k context; the 128k column shows what that shape would need.
# "MHA" and "MQA" are hypothetical variants of the same shape; the model uses 8 KV heads.
for name, kvh in [("MHA (64 KV heads)", 64), ("GQA (8 KV heads)", 8), ("MQA (1 KV head)", 1)]:
    print(f"{name:18s} | 4k tokens {kv_cache_gib(80, kvh, 128, 4096):6.2f} GiB"
          f" | 128k tokens {kv_cache_gib(80, kvh, 128, 131072):7.1f} GiB")

MHA (64 KV heads)  | 4k tokens  10.00 GiB | 128k tokens   320.0 GiB
GQA (8 KV heads)   | 4k tokens   1.25 GiB | 128k tokens    40.0 GiB
MQA (1 KV head)    | 4k tokens   0.16 GiB | 128k tokens     5.0 GiB


**Reading the output.** Decoding token by token with the cache reproduces the full forward pass up
to float32 rounding. Without a cache the time per token grows with the position, because the whole
prefix is recomputed at every step; with the cache it stays nearly flat. For the Llama-2-70B shape,
8 key/value heads instead of 64 cut the cache by a factor of 8: from 320 GiB to 40 GiB for a single
128k-token sequence.

DeepSeek-V2 and V3 go further with multi-head latent attention, which works by "significantly
compressing the Key-Value (KV) cache into a latent vector" (DeepSeek-AI, 2024a).

## 5. The FlashAttention idea: online softmax

A naive implementation of attention materialises the full $T \times T$ score matrix: for 100k
tokens, $10^{10}$ numbers per head. FlashAttention (Dao et al., 2022) is "an IO-aware exact
attention algorithm that uses tiling to reduce the number of memory reads/writes between GPU high
bandwidth memory (HBM) and GPU on-chip SRAM". It processes the matrix block by block and never
stores it in full.

What makes block-wise processing possible is an *online softmax*. For each query, keep a running
maximum $m$, a running normaliser $\ell$ and a running unnormalised output $o$. For a new block of
scores $s_j$ with values $v_j$:

$$m' = \max\big(m, \max_j s_j\big), \qquad
\ell' = e^{m - m'}\,\ell + \sum_j e^{s_j - m'}, \qquad
o' = e^{m - m'}\,o + \sum_j e^{s_j - m'}\,v_j ,$$

and the final output is $o / \ell$. This is algebraically identical to the ordinary softmax, so the
result is exact up to floating-point rounding.

The sketch below shows only this arithmetic, tiling over keys on the CPU. The real kernel also tiles
the queries and is written so that each tile fits in on-chip memory. With the CUDA backend,
`F.scaled_dot_product_attention` may dispatch to a fused FlashAttention-2 kernel (PyTorch
documentation); on the CPU, as in this notebook, it uses the plain implementation.

In [8]:
def tiled_attention(q, k, v, block=16):
    T, d = q.shape
    out = torch.zeros_like(q)
    m = torch.full((T, 1), float("-inf"))           # running max of scores, per query
    l = torch.zeros(T, 1)                           # running softmax denominator
    for s in range(0, k.shape[0], block):
        scores = q @ k[s:s + block].T / math.sqrt(d)            # only T x block at a time
        m_new = torch.maximum(m, scores.max(-1, keepdim=True).values)
        p = torch.exp(scores - m_new)
        scale = torch.exp(m - m_new)                # rescale what was accumulated so far
        l = l * scale + p.sum(-1, keepdim=True)
        out = out * scale + p @ v[s:s + block]
        m = m_new
    return out / l

q, k, v = (torch.randn(256, 64) for _ in range(3))
reference = F.softmax(q @ k.T / math.sqrt(64), dim=-1) @ v
print("max |tiled - standard|:", f"{(tiled_attention(q, k, v) - reference).abs().max().item():.1e}")
print(f"scores held in memory: standard {256 * 256:,} | tiled {256 * 16:,}")

max |tiled - standard|: 2.7e-07
scores held in memory: standard 65,536 | tiled 4,096


## 6. Mixture of Experts

A dense model uses every parameter for every token. A **sparse Mixture of Experts** layer replaces
the FFN with $n$ expert FFNs and a small **router** that sends each token to only the top-$k$ of
them (Shazeer et al., 2017; Lepikhin et al., 2020; Fedus et al., 2021). The parameter count grows
with $n$, while the compute per token depends on $k$.

**Public examples.** Mixtral 8x7B has 8 experts per layer, 2 active per token: 46.7B parameters in
total, 12.9B used per token (Mistral AI, 2023; the paper, Jiang et al., 2024, rounds to 47B and
13B). DeepSeek-V3 has 671B parameters in total and 37B activated per token; each MoE layer has
256 routed experts, 8 of them active per token, plus 1 shared expert that is always active
(DeepSeek-AI, 2024b).

**Load balancing.** Shazeer et al. (2017) observed that "the gating network tends to converge to a
state where it always produces large weights for the same few experts. This imbalance is
self-reinforcing, as the favored experts are trained more rapidly and thus are selected even more by
the gating network." The Switch Transformer (Fedus et al., 2021) adds an auxiliary loss

$$\mathcal{L}_{aux} = \alpha\, n \sum_{i=1}^{n} f_i\, P_i ,$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $P_i$ is the mean router probability
of expert $i$ over the batch; they use $\alpha = 10^{-2}$. Only $P$ carries a gradient. Under
uniform routing, $f_i = P_i = 1/n$ and the loss equals $\alpha$. If routing follows the
probabilities ($f = P$), it becomes $\alpha\, n \sum_i P_i^2 \ge \alpha$, with equality only for
the uniform distribution (Cauchy-Schwarz), which is why it "encourages uniform routing".

Switch routes each token to one expert. For top-$k$ routing the code below counts all $k$
assignments of a token and divides by $k$, the same normalisation as in the balance loss of
DeepSeek-V3. DeepSeek-V3 itself mainly uses another mechanism: a bias added to each expert's routing
score, lowered after each step if the expert is overloaded and raised if it is underloaded, together
with a sequence-wise balance loss whose weight is "an extremely small value".

**Experiment.** The second cell trains the layer on a toy regression task (16 input clusters, each
with its own target function), with and without the auxiliary loss. Every run uses the same task and
the same held-out set; the seed changes only the initialisation and the training batches.

In [9]:
class MoE(nn.Module):
    def __init__(self, dim, hidden, n_experts=8, top_k=2):
        super().__init__()
        self.n, self.k = n_experts, top_k
        self.router = nn.Linear(dim, n_experts, bias=False)
        self.experts = nn.ModuleList(SwiGLU(dim, hidden) for _ in range(n_experts))

    def forward(self, x):
        shape = x.shape
        x = x.reshape(-1, shape[-1])                           # every token is routed on its own
        probs = F.softmax(self.router(x), dim=-1)              # (tokens, n_experts)
        weights, idx = probs.topk(self.k, dim=-1)              # k experts per token
        weights = weights / weights.sum(-1, keepdim=True)      # renormalise over the chosen k
        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            tok, slot = (idx == e).nonzero(as_tuple=True)      # tokens that picked expert e
            if tok.numel():
                out.index_add_(0, tok, weights[tok, slot, None] * expert(x[tok]))
        f = F.one_hot(idx, self.n).float().sum(1).mean(0) / self.k   # fraction of tokens per expert
        P = probs.mean(0)                                            # mean router probability
        self.aux_loss = self.n * (f * P).sum()                       # load-balancing loss
        self.load = f.detach()
        return out.view(shape)

moe = MoE(dim=512, hidden=1365, n_experts=8, top_k=2)
total = count(moe)
active = count(moe.router) + 2 * count(moe.experts[0])
print(f"total params: {total:,} | active per token: {active:,} ({active / total:.0%})")

total params: 16,777,216 | active per token: 4,197,376 (25%)


In [10]:
def make_task(dim=32, clusters=16, task_seed=1):
    g = torch.Generator().manual_seed(task_seed)              # the same task in every run
    centers = torch.randn(clusters, dim, generator=g) * 2
    maps = torch.randn(clusters, dim, dim, generator=g) / math.sqrt(dim)
    def batch(n, gen):
        c = torch.randint(clusters, (n,), generator=gen)
        x = centers[c] + 0.3 * torch.randn(n, dim, generator=gen)
        return x, torch.tanh(torch.einsum("nd,nde->ne", x, maps[c]))
    return batch

def train_moe(aux_coef, seed, steps=800):
    torch.manual_seed(seed)                                   # model initialisation
    batch = make_task()
    gen = torch.Generator().manual_seed(1000 + seed)          # training batches
    model = MoE(dim=32, hidden=32, n_experts=8, top_k=2)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for step in range(steps):
        x, y = batch(512, gen)
        loss = F.mse_loss(model(x), y)
        (loss + aux_coef * model.aux_loss).backward()
        opt.step(); opt.zero_grad()
    with torch.no_grad():                                     # the same held-out set in every run
        x, y = batch(8192, torch.Generator().manual_seed(999))
        test = F.mse_loss(model(x), y).item()
    return model.load.sort(descending=True).values * model.n, test   # load in units of fair share

SEEDS = range(5)
busiest = {}
for name, coef in [("no balancing", 0.0), ("balancing loss 0.01", 0.01)]:
    runs = [train_moe(coef, seed) for seed in SEEDS]
    loads = torch.stack([r[0] for r in runs])                 # (seeds, experts), sorted per run
    tests = torch.tensor([r[1] for r in runs])
    busiest[name] = loads[:, 0]
    print(f"{name}: expert load on held-out data, sorted, in units of the fair share (1/8)")
    print("  mean over seeds:", " ".join(f"{v:.2f}" for v in loads.mean(0).tolist()))
    print("  std over seeds: ", " ".join(f"{v:.2f}" for v in loads.std(0).tolist()))
    print(f"  busiest expert {loads[:, 0].mean():.2f} ± {loads[:, 0].std():.2f} | "
          f"least used {loads[:, -1].mean():.2f} ± {loads[:, -1].std():.2f} | "
          f"held-out MSE {tests.mean():.4f} ± {tests.std():.4f}")
wins = (busiest["balancing loss 0.01"] < busiest["no balancing"]).sum().item()
print(f"busiest expert is less loaded with the balancing loss in {wins} of {len(SEEDS)} seeds")

no balancing: expert load on held-out data, sorted, in units of the fair share (1/8)
  mean over seeds: 1.60 1.32 1.25 0.97 0.80 0.75 0.71 0.59
  std over seeds:  0.24 0.09 0.02 0.11 0.06 0.01 0.09 0.14
  busiest expert 1.60 ± 0.24 | least used 0.59 ± 0.14 | held-out MSE 0.0056 ± 0.0000


balancing loss 0.01: expert load on held-out data, sorted, in units of the fair share (1/8)
  mean over seeds: 1.26 1.12 1.07 1.02 0.95 0.92 0.84 0.82
  std over seeds:  0.04 0.06 0.04 0.04 0.03 0.03 0.03 0.03
  busiest expert 1.26 ± 0.04 | least used 0.82 ± 0.03 | held-out MSE 0.0058 ± 0.0003
busiest expert is less loaded with the balancing loss in 5 of 5 seeds


**Reading the output.** Without the auxiliary loss the load is uneven: averaged over 5 seeds, the
busiest expert receives about 1.6 times its fair share and the least used about 0.6 times. With the
Switch loss at $\alpha = 10^{-2}$ the spread narrows to about 1.3 and 0.8, the busiest expert is
less loaded in all 5 seeds, and the held-out error is essentially unchanged. The imbalance in this
small task is mild compared with the collapse described by Shazeer et al. (2017); the experiment
shows the direction of the effect, not its size in a large model.

## 7. Speculative decoding

Sampling from a large model needs one forward pass per token. Speculative decoding (Leviathan et
al., 2023; Chen et al., 2023) lets a small **draft** model $q$ propose $\gamma$ tokens, which the
large **target** model $p$ then scores in a single parallel pass. Going left to right, a drafted
token $x$ is accepted with probability $\min\!\big(1, p(x)/q(x)\big)$. At the first rejection, that
position is resampled from the normalised residual $\max(p - q, 0)$ and the remaining drafts are
discarded. If all $\gamma$ drafts are accepted, the same pass yields one extra token from $p$.

**Why the output is exact.** Let $\beta = \sum_x \min\big(p(x), q(x)\big)$ be the acceptance
probability (Leviathan et al., Theorem 3.5). The residual normalises to $1 - \beta$, so the
probability of emitting $x$ is

$$q(x)\min\!\Big(1, \frac{p(x)}{q(x)}\Big) + (1-\beta)\,\frac{p(x) - \min\big(p(x), q(x)\big)}{1-\beta}
= \min\big(p(x), q(x)\big) + p(x) - \min\big(p(x), q(x)\big) = p(x) .$$

**Speed-up.** If acceptances are i.i.d. with rate $\alpha = \mathbb{E}[\beta]$, one target pass
produces $(1 - \alpha^{\gamma+1})/(1 - \alpha)$ tokens in expectation (Leviathan et al., Eq. 1). The
draft model's own cost comes on top of that. The cell below checks the acceptance rule empirically
for a single position.

In [11]:
def speculative_sample(p, q, n):
    x = torch.multinomial(q, n, replacement=True)      # draft proposals
    accept = torch.rand(n) < (p[x] / q[x]).clamp(max=1)
    residual = (p - q).clamp(min=0)
    residual /= residual.sum()
    fallback = torch.multinomial(residual, n, replacement=True)
    y = torch.where(accept, x, fallback)
    return y, accept.float().mean().item()

torch.manual_seed(0)
logits = torch.randn(8) * 1.5
p = F.softmax(logits, 0)                                  # big "target" model
q = F.softmax(logits + 0.8 * torch.randn(8), 0)           # small, imperfect draft model
n = 400_000
y, acc = speculative_sample(p, q, n)
emp = torch.bincount(y, minlength=8).float() / n
print("target p   :", " ".join(f"{v:.3f}" for v in p.tolist()))
print("draft q    :", " ".join(f"{v:.3f}" for v in q.tolist()))
print("speculative:", " ".join(f"{v:.3f}" for v in emp.tolist()))
print(f"acceptance rate {acc:.3f}  (theory: sum min(p, q) = {torch.minimum(p, q).sum():.3f})")

tv = lambda a, b: 0.5 * (a - b).abs().sum().item()        # total variation distance
noise = 0.5 * (2 * p * (1 - p) / (math.pi * n)).sqrt().sum().item()
print(f"total variation distance to p: draft {tv(q, p):.4f} | speculative output {tv(emp, p):.4f} "
      f"(expected from sampling noise alone: {noise:.4f})")

print("\nexpected tokens per big-model pass, drafting gamma = 4 tokens")
print("(assuming each draft token is accepted independently with the same probability):")
for a in [0.6, 0.7, 0.8, 0.9]:
    print(f"  acceptance {a:.1f} -> {(1 - a**5) / (1 - a):.2f} tokens")

target p   : 0.537 0.034 0.002 0.125 0.010 0.007 0.097 0.187
draft q    : 0.407 0.033 0.002 0.195 0.007 0.021 0.056 0.278
speculative: 0.537 0.034 0.002 0.125 0.010 0.006 0.098 0.187
acceptance rate 0.824  (theory: sum min(p, q) = 0.824)
total variation distance to p: draft 0.1759 | speculative output 0.0005 (expected from sampling noise alone: 0.0012)

expected tokens per big-model pass, drafting gamma = 4 tokens
(assuming each draft token is accepted independently with the same probability):
  acceptance 0.6 -> 2.31 tokens
  acceptance 0.7 -> 2.77 tokens
  acceptance 0.8 -> 3.36 tokens
  acceptance 0.9 -> 4.10 tokens


## 8. Quantization

Weights are trained in 16 or 32 bits but can be stored with fewer. *Absmax* quantization maps a
group of weights to the integers $\{-q_{max}, \dots, q_{max}\}$ using one scale, $\max|w| / q_{max}$,
per group. A single large weight forces a large scale on its whole group and wastes precision for
all the others; smaller groups confine the damage, at the cost of storing more scales.

The demo uses a **synthetic** weight matrix (Gaussian, with 0.1% of entries multiplied by 8), so the
error values illustrate the mechanism and are not predictions for a real model. In real LLMs the
strongest outliers are in the activations: Dettmers et al. (2022) isolate "outlier feature
dimensions" in a 16-bit multiplication while "more than 99.9% of values are multiplied in 8-bit".
Practical low-bit methods do much better than the round-to-nearest used here: GPTQ (Frantar et al.,
2023) is "based on approximate second-order information" and reaches "3 or 4 bits per weight, with
negligible accuracy degradation", and AWQ (Lin et al., 2024) protects the "1% salient weights",
identified from activation statistics, by rescaling their channels. At 4 bits per weight a
70B-parameter model shrinks from about 140 GB in fp16 to about 35 GB.

In [12]:
def quantize(w, bits, group=None):
    qmax = 2 ** (bits - 1) - 1
    g = w.reshape(-1, group) if group else w                  # per-row, or per-group of weights
    scale = g.abs().amax(dim=1, keepdim=True) / qmax
    q = torch.round(g / scale).clamp(-qmax, qmax)             # small integers
    return (q * scale).reshape(w.shape)                       # dequantized weights

torch.manual_seed(0)
W = torch.randn(1024, 1024) * 0.02
W[torch.rand_like(W) < 0.001] *= 8                            # a few large-magnitude weights (outliers)
x = torch.randn(64, 1024)
ref = x @ W.T
print(f"{'format':22s} {'MB':>6s} {'output error':>13s}")
print(f"{'fp16':22s} {W.numel() * 2 / 1e6:6.2f} {'0':>13s}")
for bits, group in [(8, None), (4, None), (4, 64), (3, 64), (2, 64)]:
    err = ((x @ quantize(W, bits, group).T - ref).norm() / ref.norm()).item()
    mb = (W.numel() * bits / 8 + (W.numel() / (group or 1024)) * 2) / 1e6
    label = f"int{bits} " + (f"group {group}" if group else "per-row")
    print(f"{label:22s} {mb:6.2f} {err:12.1%}")

format                     MB  output error
fp16                     2.10             0
int8 per-row             1.05         1.7%
int4 per-row             0.53        30.9%
int4 group 64            0.56        12.9%
int3 group 64            0.43        27.9%
int2 group 64            0.29        70.8%


**Reading the output.** The error is the relative error of the layer's output on random inputs; the
size includes one fp16 scale per group. On this synthetic matrix, 8 bits per weight are nearly
lossless. At 4 bits, one scale per row loses much more than one scale per 64 weights, because each
row contains about one outlier.

## 9. Post-training: preference optimization

A pre-trained model predicts text; it is not yet an assistant. Post-training typically adds
(1) **supervised fine-tuning** on example conversations and (2) learning from **preferences**
between pairs of responses.

**RLHF** fits a reward model to human preference labels and then optimises the policy against it
with reinforcement learning (Christiano et al., 2017, for RL agents; Ouyang et al., 2022, and
Bai et al., 2022a, for language models).

**Constitutional AI** (Bai et al., 2022b) trains for harmlessness "without any human labels
identifying harmful outputs. The only human oversight is provided through a list of rules or
principles". In a supervised phase the model critiques and revises its own responses and is
fine-tuned on the revisions. In a reinforcement-learning phase a model judges which of two samples
is better according to the principles, a preference model is trained on these AI preferences, and
it provides the reward: "RL from AI Feedback". The authors still "use human labels for helpfulness,
but only AI labels for harmlessness".

**DPO** (Rafailov et al., 2023) removes the separate reward model and the RL loop. It shows that the
RLHF objective can be optimised "with only a simple classification loss":

$$\mathcal{L}_{DPO} = -\,\mathbb{E}_{(x,\,y_w,\,y_l)}\left[\log \sigma\!\left(
\beta \log\frac{\pi_\theta(y_w \mid x)}{\pi_{ref}(y_w \mid x)}
- \beta \log\frac{\pi_\theta(y_l \mid x)}{\pi_{ref}(y_l \mid x)}\right)\right] ,$$

where $y_w$ is the preferred response, $y_l$ the rejected one, and $\pi_{ref}$ a frozen copy of the
initial model.

Reinforcement learning is also used after pre-training to improve reasoning. DeepSeek-R1
(DeepSeek-AI, 2025) uses rule-based rewards on problems whose answers can be checked, in
mathematics, code and logic.

In [13]:
def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    # inputs: log-probability of each whole response,
    # under the policy (pi) and a frozen reference model (ref)
    chosen_gain = pi_chosen - ref_chosen
    rejected_gain = pi_rejected - ref_rejected
    margin = beta * (chosen_gain - rejected_gain)
    return -F.logsigmoid(margin).mean()

ref_c, ref_r = torch.tensor([-40.0]), torch.tensor([-42.0])
for pc, pr, note in [(-40.0, -42.0, "policy = reference"),
                     (-36.0, -48.0, "prefers chosen more"),
                     (-44.0, -38.0, "prefers rejected")]:
    loss = dpo_loss(torch.tensor([pc]), torch.tensor([pr]), ref_c, ref_r)
    print(f"{note:22s} loss {loss.item():.3f}")

policy = reference     loss 0.693
prefers chosen more    loss 0.313
prefers rejected       loss 1.171


## 10. Superposition and sparse autoencoders

Individual neurons often respond to several unrelated concepts ("polysemanticity"). Elhage et al.
(2022) showed in toy models that a network can represent "more features than [it has] dimensions"
by assigning sparse features to overlapping directions, which they call *superposition*, while
cautioning that "it's very unclear what to generalize to real networks". For real networks,
superposition is a hypothesised cause of polysemanticity (Cunningham et al., 2023).

A **sparse autoencoder** (SAE) tests that hypothesis by learning an overcomplete dictionary in which
only a few latents are active for each input. Bricken et al. (2023) applied it to a one-layer
transformer, and Templeton et al. (2024) to the middle layer of Claude 3 Sonnet, with dictionaries
of about 1M, 4M and 34M features. One of those features responds to the Golden Gate Bridge;
clamping it to ten times its maximum activation "induces thematically-related model behavior".

**Experiment.** In the spirit of the synthetic experiments of Sharkey et al. (2022): 32
ground-truth features, each a random unit direction, are active with probability 0.03 and mixed
into a 16-dimensional activation vector. There are more features than dimensions, so they must overlap. An SAE with 64
latents is trained with a mean-squared reconstruction loss plus an L1 penalty, with unit-norm
decoder columns as in Bricken et al. (this SAE is simplified: it has no pre-encoder bias).

Reported metrics: the number of features whose best-matching dictionary direction has cosine
similarity above 0.9; the mean of that best cosine over features (*mean max cosine similarity*,
MMCS); the same for single neurons, taking the largest absolute coordinate of each feature
direction; the number of latents that ever fire; the mean number of active latents per input (L0)
against the true number of active features; and the fraction of variance left unexplained.

In [14]:
n_feat, d_act, n_latent = 32, 16, 64

def run_sae(seed, steps=4000, l1=1e-2, p_active=0.03):
    torch.manual_seed(seed)
    true_dirs = F.normalize(torch.randn(n_feat, d_act), dim=1)     # one random direction per feature

    def activations(n=2048):
        mags = torch.rand(n, n_feat) * (torch.rand(n, n_feat) < p_active)
        return mags @ true_dirs, mags                               # 32 features -> 16 numbers

    enc = nn.Linear(d_act, n_latent)                # encoder: 16 -> 64 latents
    dec = nn.Linear(n_latent, d_act)                # decoder columns = the dictionary
    params = [*enc.parameters(), *dec.parameters()]
    opt = torch.optim.Adam(params, lr=3e-3)
    for step in range(steps):
        a, _ = activations()
        z = F.relu(enc(a))                          # sparse latent code
        recon = dec(z)
        recon_loss = F.mse_loss(recon, a)           # reconstruct activations
        sparsity = z.abs().sum(1).mean()            # L1: few active latents
        loss = recon_loss + l1 * sparsity
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():                       # unit-length dictionary
            dec.weight.data = F.normalize(dec.weight.data, dim=0)

    dictionary = dec.weight.detach().T                             # (n_latent, d_act), unit rows
    best_cos = (true_dirs @ dictionary.T).max(1).values            # signed: same direction required
    neuron_cos = true_dirs.abs().max(1).values                     # best single neuron, either sign
    with torch.no_grad():
        a, mags = activations(8192)
        z = F.relu(enc(a))
        fvu = ((dec(z) - a).pow(2).sum() / (a - a.mean(0)).pow(2).sum()).item()
    fired = z > 1e-3
    return dict(recovered=(best_cos > 0.9).sum().item(), mmcs=best_cos.mean().item(),
                neurons=(neuron_cos > 0.9).sum().item(), neuron_cos=neuron_cos.mean().item(),
                alive=fired.any(0).sum().item(), l0=fired.float().sum(1).mean().item(),
                true_l0=(mags > 0).float().sum(1).mean().item(), fvu=fvu)

runs = [run_sae(seed) for seed in range(5)]
for seed, r in enumerate(runs):
    print(f"seed {seed}: SAE recovers {r['recovered']}/{n_feat} features (MMCS {r['mmcs']:.3f}) | "
          f"single neurons {r['neurons']}/{n_feat} ({r['neuron_cos']:.2f}) | "
          f"alive latents {r['alive']}/{n_latent} | L0 {r['l0']:.2f} (true {r['true_l0']:.2f}) | "
          f"unexplained variance {r['fvu']:.1%}")
mean = lambda key: sum(r[key] for r in runs) / len(runs)
std = lambda key: torch.tensor([float(r[key]) for r in runs]).std().item()
print(f"mean over {len(runs)} seeds: SAE {mean('recovered'):.1f} ± {std('recovered'):.1f} of {n_feat} "
      f"(MMCS {mean('mmcs'):.3f}) | single neurons {mean('neurons'):.1f} ± {std('neurons'):.1f} "
      f"of {n_feat} ({mean('neuron_cos'):.2f})")

seed 0: SAE recovers 32/32 features (MMCS 0.981) | single neurons 0/32 (0.55) | alive latents 56/64 | L0 1.90 (true 0.97) | unexplained variance 7.5%
seed 1: SAE recovers 32/32 features (MMCS 0.990) | single neurons 0/32 (0.53) | alive latents 56/64 | L0 1.67 (true 0.96) | unexplained variance 7.5%
seed 2: SAE recovers 30/32 features (MMCS 0.984) | single neurons 0/32 (0.52) | alive latents 50/64 | L0 1.55 (true 0.96) | unexplained variance 7.2%
seed 3: SAE recovers 32/32 features (MMCS 0.982) | single neurons 0/32 (0.53) | alive latents 54/64 | L0 1.83 (true 0.95) | unexplained variance 7.3%
seed 4: SAE recovers 32/32 features (MMCS 0.982) | single neurons 0/32 (0.54) | alive latents 53/64 | L0 1.79 (true 0.97) | unexplained variance 7.0%
mean over 5 seeds: SAE 31.6 ± 0.9 of 32 (MMCS 0.984) | single neurons 0.0 ± 0.0 of 32 (0.53)


**Reading the output.** No feature is aligned with a single neuron in any seed, while the SAE
recovers 30 to 32 of the 32 features, with a mean best cosine of about 0.98. The recovery is not
perfect in other respects: some latents never fire, about two latents are active per input although
on average only one feature is, and several percent of the variance is left unexplained. The last
effect is expected from the L1 penalty, which shrinks the latent activations. Ground truth is known
here by construction; in a real model it is not, which is why interpreting SAE features there
requires separate evidence.

## Scope and limitations

- The experiments are small, on synthetic data, and on a CPU. They demonstrate that each mechanism
  works as described; they do not measure its benefit in a large model.
- Quoted figures for specific models (parameter counts, token counts, head counts) are those
  reported by their authors in the cited sources and were not measured here.
- The post-training section describes published methods. How any particular closed model is trained
  is not public beyond what its developer has published.

## References

**Scaling**
- Kaplan et al. (2020). *Scaling Laws for Neural Language Models.* [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
- Brown et al. (2020). *Language Models are Few-Shot Learners.* [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
- Hoffmann et al. (2022). *Training Compute-Optimal Large Language Models.* [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
- Besiroglu et al. (2024). *Chinchilla Scaling: A replication attempt.* [arXiv:2404.10102](https://arxiv.org/abs/2404.10102)
- Sardana et al. (2024). *Beyond Chinchilla-Optimal: Accounting for Inference in Language Model Scaling Laws.* ICML 2024. [arXiv:2401.00448](https://arxiv.org/abs/2401.00448)
- Touvron et al. (2023). *Llama 2: Open Foundation and Fine-Tuned Chat Models.* [arXiv:2307.09288](https://arxiv.org/abs/2307.09288)
- Grattafiori et al. (2024). *The Llama 3 Herd of Models.* [arXiv:2407.21783](https://arxiv.org/abs/2407.21783)
- Meta (2024). *Llama 3 model card.* [github.com/meta-llama/llama3](https://github.com/meta-llama/llama3/blob/main/MODEL_CARD.md)

**Architecture and inference**
- Zhang & Sennrich (2019). *Root Mean Square Layer Normalization.* NeurIPS 2019. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467)
- Shazeer (2020). *GLU Variants Improve Transformer.* [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- Shazeer (2019). *Fast Transformer Decoding: One Write-Head is All You Need.* [arXiv:1911.02150](https://arxiv.org/abs/1911.02150)
- Ainslie et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints.* EMNLP 2023. [arXiv:2305.13245](https://arxiv.org/abs/2305.13245)
- Dao et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness.* [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
- PyTorch documentation. [`torch.nn.functional.scaled_dot_product_attention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- Leviathan et al. (2023). *Fast Inference from Transformers via Speculative Decoding.* ICML 2023. [arXiv:2211.17192](https://arxiv.org/abs/2211.17192)
- Chen et al. (2023). *Accelerating Large Language Model Decoding with Speculative Sampling.* [arXiv:2302.01318](https://arxiv.org/abs/2302.01318)

**Mixture of Experts**
- Shazeer et al. (2017). *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer.* [arXiv:1701.06538](https://arxiv.org/abs/1701.06538)
- Lepikhin et al. (2020). *GShard: Scaling Giant Models with Conditional Computation and Automatic Sharding.* [arXiv:2006.16668](https://arxiv.org/abs/2006.16668)
- Fedus et al. (2021). *Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity.* JMLR. [arXiv:2101.03961](https://arxiv.org/abs/2101.03961)
- Mistral AI (2023). *Mixtral of experts* (announcement). [mistral.ai/news/mixtral-of-experts](https://mistral.ai/news/mixtral-of-experts/)
- Jiang et al. (2024). *Mixtral of Experts.* [arXiv:2401.04088](https://arxiv.org/abs/2401.04088)
- DeepSeek-AI (2024a). *DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model.* [arXiv:2405.04434](https://arxiv.org/abs/2405.04434)
- DeepSeek-AI (2024b). *DeepSeek-V3 Technical Report.* [arXiv:2412.19437](https://arxiv.org/abs/2412.19437)

**Quantization**
- Dettmers et al. (2022). *LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale.* NeurIPS 2022. [arXiv:2208.07339](https://arxiv.org/abs/2208.07339)
- Frantar et al. (2023). *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers.* ICLR 2023. [arXiv:2210.17323](https://arxiv.org/abs/2210.17323)
- Lin et al. (2024). *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration.* MLSys 2024. [arXiv:2306.00978](https://arxiv.org/abs/2306.00978)

**Post-training**
- Christiano et al. (2017). *Deep reinforcement learning from human preferences.* [arXiv:1706.03741](https://arxiv.org/abs/1706.03741)
- Ouyang et al. (2022). *Training language models to follow instructions with human feedback.* [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
- Bai et al. (2022a). *Training a Helpful and Harmless Assistant with Reinforcement Learning from Human Feedback.* [arXiv:2204.05862](https://arxiv.org/abs/2204.05862)
- Bai et al. (2022b). *Constitutional AI: Harmlessness from AI Feedback.* [arXiv:2212.08073](https://arxiv.org/abs/2212.08073)
- Rafailov et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.* [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
- DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.* Nature 645, 633-638. [arXiv:2501.12948](https://arxiv.org/abs/2501.12948)

**Interpretability**
- Elhage et al. (2022). *Toy Models of Superposition.* [arXiv:2209.10652](https://arxiv.org/abs/2209.10652)
- Sharkey, Braun & beren (2022). *[Interim research report] Taking features out of superposition with sparse autoencoders.* [AI Alignment Forum](https://www.alignmentforum.org/posts/z6QQJbtpkEAX3Aojj/interim-research-report-taking-features-out-of-superposition)
- Cunningham et al. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models.* [arXiv:2309.08600](https://arxiv.org/abs/2309.08600)
- Bricken et al. (2023). *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning.* [transformer-circuits.pub](https://transformer-circuits.pub/2023/monosemantic-features/index.html)
- Templeton et al. (2024). *Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet.* [transformer-circuits.pub](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)